# AVA Spatial Agent Integration

Combines DroneKit control with GeoJSON spatial queries.
The drone can now find and navigate to conservation targets.

In [1]:
!pip install -q agno dronekit pymavlink pydantic


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os

if not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = input('OpenRouter API Key: ')

print('API key set.' if os.environ.get('OPENROUTER_API_KEY') else 'No API key!')

API key set.


In [6]:
from dronekit import connect, VehicleMode
import time

CONNECTION_STRING = 'tcp:127.0.0.1:5763'

print('Connecting to vehicle...')
vehicle = connect(CONNECTION_STRING, wait_ready=True, baud=57600, rate=60)
print(f'Connected!')
print(f'  Mode:     {vehicle.mode.name}')
print(f'  GPS:      {vehicle.gps_0}')
print(f'  Position: {vehicle.location.global_frame.lat}, {vehicle.location.global_frame.lon}')

Connecting to vehicle...
Connected!
  Mode:     STABILIZE
  GPS:      GPSInfo:fix=6,num_sat=10
  Position: -35.3633516, 149.1652413


## Import Spatial and Drone Tools

In [8]:
from agno.tools import tool
from agents.spatial_agent import spatial_tools

# Basic drone control tools
@tool
def get_current_position() -> str:
    """
    Get current GPS coordinates of the drone.
    """
    lat = vehicle.location.global_frame.lat
    lon = vehicle.location.global_frame.lon
    alt = vehicle.location.global_relative_frame.alt
    return f"Lat: {lat}, Lon: {lon}, Alt: {alt}m"

@tool
def go_to_coords(lat: float, lon: float, alt: float = None) -> str:
    """
    Fly to GPS coordinates.
    Args:
        lat: Destination latitude
        lon: Destination longitude
        alt: Destination altitude in meters (optional)
    """
    from dronekit import LocationGlobalRelative
    
    if alt is None:
        alt = vehicle.location.global_relative_frame.alt
    
    target = LocationGlobalRelative(lat, lon, alt)
    vehicle.simple_goto(target)
    
    return f"Flying to {lat}, {lon} at {alt}m"

# Combine all tools
all_tools = spatial_tools + [get_current_position, go_to_coords]

print(f"Loaded {len(all_tools)} tools:")
for t in all_tools:
    print(f"  - {t.name}")

Loaded 5 tools:
  - get_nearest_target
  - get_target_by_name
  - list_all_targets
  - get_current_position
  - go_to_coords


## Build Combined Agent

In [10]:
from agno.agent import Agent  # Note: agno.agent not just agno
from agno.models.openrouter import OpenRouter

SYSTEM_PROMPT = """
You are AVA - an autonomous assistant for conservation drone operations.

You can:
- Query conservation targets from the GeoJSON database
- Find nearest targets to the drone's position
- Navigate the drone to target locations
- Provide target metadata and observation history

When asked to find or visit a target:
1. Get current drone position
2. Query for nearest target (or lookup by name)
3. Navigate to target coordinates
4. Report target details

Be concise and operational in tone.
"""

agent = Agent(
    name='AVA_Combined',
    model=OpenRouter(
        id='google/gemini-2.0-flash-001',
        api_key=os.environ['OPENROUTER_API_KEY'],
    ),
    instructions=SYSTEM_PROMPT,
    tools=all_tools,
    add_history_to_context=True,
    debug_mode=True
)

print('Agent ready.')

Agent ready.


## Test Queries

In [11]:
# Test 1: Find nearest target to current position
result = agent.run("What's the nearest conservation target to my current position?")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: f25073e9-8fac-4d85-8323-ae827c72d7c2

DEBUG ** Agent Run Start: 6a544eb5-bd71-42a9-9f4f-83b5c9fa1026 ***

DEBUG Processing tools for model

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 6a544eb5-bd71-42a9-9f4f-83b5c9fa1026 ****

The nearest conservation target is Alpine Creek Water Source (target_002), located 13638.9km away at 40.018, -105.282. It's a medium priority water source that hasn't been visited yet.



In [12]:
# Test 2: Lookup specific target
result = agent.run("Tell me about Rocky Ridge Wildlife Habitat")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: f25073e9-8fac-4d85-8323-ae827c72d7c2 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: f25073e9-8fac-4d85-8323-ae827c72d7c2

DEBUG ** Agent Run Start: 93460385-8e4b-4bf3-829b-4d8cc03e0735 ***

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 93460385-8e4b-4bf3-829b-4d8cc03e0735 ****

Target not found. Did you mean Rocky Ridge? I can search for that target instead.



In [13]:
# Test 3: Navigate to nearest target
result = agent.run("Take me to the nearest high priority target")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: f25073e9-8fac-4d85-8323-ae827c72d7c2 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: f25073e9-8fac-4d85-8323-ae827c72d7c2

DEBUG ** Agent Run Start: 0ed125a7-9da8-4a8e-9ebb-5fe3ae4a4366 ***

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 0ed125a7-9da8-4a8e-9ebb-5fe3ae4a4366 ****

Navigating to Alpine Creek Water Source at 40.018, -105.282. Distance is 13638.9km.
Flying to Alpine Creek Water Source at 40.018, -105.282 at 2380.0m.


In [14]:
# Test 4: List unvisited targets
result = agent.run("Show me all targets that haven't been visited yet")
print(result.content)

WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: f25073e9-8fac-4d85-8323-ae827c72d7c2 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: f25073e9-8fac-4d85-8323-ae827c72d7c2

DEBUG ** Agent Run Start: 90601590-e260-48f6-b259-3ec985a67210 ***

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 90601590-e260-48f6-b259-3ec985a67210 ****

Here are the targets that haven't been visited: Alpine Creek Water Source, and South Slope Aspen Grove.


## Interactive Mode

In [15]:
print('AVA Spatial Agent - Interactive Mode')
print('Type commands. Type "exit" to quit.\n')

while True:
    user_input = input('\nCommand: ')
    
    if user_input.lower().strip() == 'exit':
        break
    
    if not user_input.strip():
        continue
    
    try:
        result = agent.run(user_input)
        print(f'\nResponse: {result.content}')
    except Exception as e:
        print(f'\nERROR: {e}')

print('Session ended.')

AVA Spatial Agent - Interactive Mode
Type commands. Type "exit" to quit.



WARNING  add_history_to_context is True, but no database has been assigned to the agent. History will not be added 
         to the context.

DEBUG ***** Session ID: f25073e9-8fac-4d85-8323-ae827c72d7c2 *****

DEBUG ****************** Agent ID: ava-combined ******************

DEBUG Creating new AgentSession: f25073e9-8fac-4d85-8323-ae827c72d7c2

...

DEBUG ************************  METRICS  *************************

DEBUG ----------------- OpenRouter Response End ------------------

DEBUG Added RunOutput to Agent Session

DEBUG *** Agent Run End: 3dff0908-e7da-4453-a2d3-862351c9675c ****


Response: Sorry, I cannot plan a route to visit all unvisited targets. I can only find the nearest target or go to a specific target by name or coordinates. Could you please provide the current drone position or a target name to start with?

Session ended.


In [ ]:
vehicle.close()
print('Vehicle connection closed.')